# Savant 00, baseline before any training

This notebook measures the untrained base model inside the Savant harness, so later training has an honest starting point. It runs on the held-out bench tables, which are never used for tuning.

1. The guarded statistics engine runs alone on every table, which needs no model.
2. The base model then answers one open question per table through the agent loop, and every number it cites must come from code it ran.

In [ ]:
import os

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import glob
import json
import subprocess
import sys
import time

wheel = sorted(glob.glob("/kaggle/input/**/savant-*.whl", recursive=True))[-1]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", wheel], check=True)
TASKS = sorted(glob.glob("/kaggle/input/**/tasks/*.csv", recursive=True))
print(wheel, len(TASKS), "tables")

In [ ]:
# Settings for this run. The model can be swapped for Qwen/Qwen3.5-4B once transformers on Kaggle supports it.
MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"
AGENT_TABLES = 10  # tables given to the model, half null and half planted
QUESTION = (
    "What are the most important relationships in this data? Report only what the data supports."
)
MAX_NEW_TOKENS = 768  # longest single model reply
KEEP_MESSAGES = 6  # recent messages the model sees each round

In [ ]:
from pathlib import Path

import polars as pl

from savant.engine import Settings, analyze
from savant.grade import grade, summarize
from savant.planted import Plant, Task


def load_task(csv):
    truth = json.loads(Path(csv.replace(".csv", ".truth.json")).read_text())
    plants = [Plant(p["kind"], tuple(p["columns"]), p["strength"]) for p in truth["plants"]]
    return Task(truth["id"], pl.read_csv(csv, try_parse_dates=True), plants)


tasks = [load_task(c) for c in TASKS]
t0 = time.time()
engine_scores = [grade(t, analyze(t.table, Settings(stability_runs=100))) for t in tasks]
engine_summary = summarize(engine_scores)
print(f"{time.time() - t0:.0f}s")
print(json.dumps(engine_summary, indent=1))

In [ ]:
import torch

from savant.agent import ask
from savant.llm import Transformers

client = Transformers(MODEL_ID, max_new_tokens=MAX_NEW_TOKENS)
nulls = [t for t in tasks if t.is_null][: AGENT_TABLES // 2]
planted = [t for t in tasks if not t.is_null][: AGENT_TABLES // 2]
log = Path("/kaggle/working/runs.jsonl")
log.write_text("")
runs = []
for t in nulls + planted:
    t0 = time.time()
    try:
        r = ask(t.table, QUESTION, client, keep_messages=KEEP_MESSAGES)
        error = ""
    except torch.cuda.OutOfMemoryError as e:
        r, error = None, f"out of memory: {e}"[:300]
    torch.cuda.empty_cache()
    text = ((r.rendered if r else "") or "").lower()
    real = [p for p in t.plants if p.kind != "leak"]
    mentioned = sum(all(c.lower() in text for c in p.columns) for p in real)
    row = {
        "task": t.id,
        "null": t.is_null,
        "grounded": bool(r and r.grounded),
        "rounds": r.rounds if r else None,
        "steps": len(r.steps) if r else 0,
        "failed_steps": sum(not s.ok for s in r.steps) if r else 0,
        "planted": len(real),
        "mentioned": mentioned,
        "seconds": round(time.time() - t0, 1),
        "error": error,
        "answer": r.rendered if r else None,
        "transcript": r.transcript if r else None,
    }
    runs.append(row)
    with log.open("a") as f:
        f.write(json.dumps(row, default=str) + "\n")
    status = "grounded" if row["grounded"] else "NOT grounded"
    print(
        t.id,
        status,
        row["rounds"],
        "rounds",
        mentioned,
        "of",
        len(real),
        row["seconds"],
        "s",
        error,
    )

In [ ]:
def share(xs):
    return round(sum(xs) / len(xs), 3) if xs else None


agent_summary = {
    "model": MODEL_ID,
    "tables": len(runs),
    "grounded_answers": share([r["grounded"] for r in runs]),
    "mean_rounds": share([r["rounds"] for r in runs]),
    "failed_step_share": round(
        sum(r["failed_steps"] for r in runs) / max(1, sum(r["steps"] for r in runs)), 3
    ),
    "planted_pairs_mentioned": share([r["mentioned"] / r["planted"] for r in runs if r["planted"]]),
}
Path("/kaggle/working/baseline.json").write_text(
    json.dumps(
        {"engine": engine_summary, "agent": agent_summary, "runs": runs}, indent=1, default=str
    )
)
print(json.dumps(agent_summary, indent=1))